# Fase 2 -- Routing y calculo de tiempos de viaje

Orquesta `src/routing.py`. Toda la logica reutilizable (extraccion OSM,
construccion de grafos, snapping, matrices) vive en el modulo -- este
notebook solo llama funciones, muestra resultados y guarda las salidas.

**Motor**: `pyosmium` + `NetworkX` (no OSRM en Docker ni `pyrosm` -- ver el
docstring de `src/routing.py` para el porque). Parametros en `config.md`.

**Cacheo real**: cada paso caro (extraccion del `.pbf`, cada matriz de
ruteo) se guarda en `data/processed/` o `data/raw/osm_extract/` y se
reutiliza si ya existe -- una segunda corrida de este notebook no vuelve a
calcular nada, solo relee parquet.

In [1]:
import os
import sys
import time

import geopandas as gpd
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "data" else os.getcwd()
sys.path.insert(0, BASE)
from src import routing as R

PROCESSED_DIR = os.path.join(BASE, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE, "data", "outputs")
LOGS_DIR = os.path.join(BASE, "logs")
BOUNDARIES_PATH = os.path.join(BASE, "data", "raw", "boundaries", "peru_distrital_simple.geojson")
OSM_EXTRACT_DIR = os.path.join(BASE, "data", "raw", "osm_extract")
PBF_PATH = os.path.join(BASE, "data", "peru-260905.osm.pbf")
POP_PATH = os.path.join(BASE, "data", "raw", "ubigeo_poblacion_distrital.csv")

for d in (PROCESSED_DIR, OUTPUTS_DIR, LOGS_DIR, OSM_EXTRACT_DIR):
    os.makedirs(d, exist_ok=True)

DEPARTAMENTOS = ["LAMBAYEQUE", "JUNIN", "LORETO"]  # ver config.md
CAP_DEMANDA = 5000
SEED = 42

t_inicio = time.time()
R.log("=== FASE 2: ROUTING Y TIEMPOS DE VIAJE ===")


[17:27:41] === FASE 2: ROUTING Y TIEMPOS DE VIAJE ===


## 1. Cargar salidas de Fase 1 y población distrital (INEI)

SIGMED no trae población por centro poblado, así que se usa la población
**distrital** del censo 2017 (INEI, vía `geodir/ubigeo-peru`, descarga
cacheada) repartida en partes iguales entre los centros poblados del mismo
distrito -- limitación declarada, no la distribución real.

In [2]:
oferta = gpd.read_parquet(os.path.join(PROCESSED_DIR, "establecimientos_salud.parquet"))
demanda = gpd.read_parquet(os.path.join(PROCESSED_DIR, "centros_poblados_demanda.parquet"))
R.log(f"Oferta cargada: {len(oferta)} filas | Demanda cargada: {len(demanda)} filas")

pop = pd.read_csv(POP_PATH, encoding="utf-8-sig", dtype={"Ubigeo": str})
pop["Poblacion"] = pop["Poblacion"].astype(str).str.replace(",", "").astype(float)
pop_completa = pop.rename(columns={"Ubigeo": "UBIGEO"})
pop_join = pop_completa[["UBIGEO", "Poblacion"]]

demanda = demanda.merge(pop_join, on="UBIGEO", how="left")
n_sin_pop = demanda["Poblacion"].isna().sum()
R.log(f"Centros poblados sin poblacion distrital emparejada: {n_sin_pop} / {len(demanda)}")

cp_por_distrito = demanda.groupby("UBIGEO")["CODCP"].transform("count")
demanda["POBLACION_CP"] = (demanda["Poblacion"] / cp_por_distrito).fillna(0)

# Urbano = capital de distrito, provincia o departamento (CAPITAL != 0 en
# SIGMED) -- regla explicita, ver src/metrics.py para el uso en Fase 3.
demanda["ES_URBANO"] = demanda["CAPITAL"].astype(str) != "0"
R.log(f"Centros poblados urbanos (capital distrital/provincial/departamental): {demanda['ES_URBANO'].sum()}")

pop_completa.to_csv(os.path.join(OUTPUTS_DIR, "poblacion_distrital_inei.csv"), index=False)


[17:27:42] Oferta cargada: 2317 filas | Demanda cargada: 13358 filas


[17:27:42] Centros poblados sin poblacion distrital emparejada: 0 / 13358


[17:27:42] Centros poblados urbanos (capital distrital/provincial/departamental): 215


## 2. Muestreo de demanda: estratificado por distrito, ponderado por población

El enunciado pide tope de 5,000 puntos de demanda y una estrategia de
muestreo documentada. Con 13,358 centros poblados en los 3 departamentos,
se aplica una fracción de muestreo uniforme (~37%) **dentro de cada
distrito**, de modo que la distribución de la muestra entre distritos siga
siendo representativa. Implicación en el margen de error: un distrito con
pocos centros poblados en la muestra tendrá una estimación más ruidosa de
su acceso ponderado (se declara en el reporte de Fase 5).

In [3]:
cache_muestra = os.path.join(PROCESSED_DIR, "demanda_muestra.parquet")
if os.path.exists(cache_muestra):
    R.log("Muestra de demanda ya cacheada -> se reutiliza")
    demanda_muestra = gpd.read_parquet(cache_muestra)
else:
    if len(demanda) <= CAP_DEMANDA:
        demanda_muestra = demanda.copy()
    else:
        frac = CAP_DEMANDA / len(demanda)
        partes = []
        for ubigeo, grp in demanda.groupby("UBIGEO", group_keys=False):
            n = max(1, round(len(grp) * frac))
            n = min(n, len(grp))
            partes.append(grp.sample(n=n, random_state=SEED))
        demanda_muestra = pd.concat(partes)
        if len(demanda_muestra) > CAP_DEMANDA:
            demanda_muestra = demanda_muestra.sample(n=CAP_DEMANDA, random_state=SEED)
        demanda_muestra = gpd.GeoDataFrame(demanda_muestra, geometry="geometry", crs=demanda.crs)

        # Correccion de peso muestral (IMPORTANTE): el muestreo es
        # estratificado por distrito con una fraccion de retencion DISTINTA
        # en cada distrito (redondeo de "max(1, round(n*frac))"). Sin esto,
        # sumar POBLACION_CP sobre la muestra subestima la poblacion real en
        # ~63% (justo la fraccion que se descarto al muestrear) -- las
        # MEDIAS ponderadas seguirian siendo insesgadas, pero cualquier
        # TOTAL (poblacion cubierta, etc.) saldria mal. Se reescala cada
        # punto por (centros_poblados_totales_en_el_distrito /
        # centros_poblados_en_la_muestra_de_ese_distrito).
        n_total_distrito = demanda.groupby("UBIGEO").size()
        n_muestra_distrito = demanda_muestra.groupby("UBIGEO").size()
        factor_reponderacion = (n_total_distrito / n_muestra_distrito).rename("FACTOR_REPONDERACION")
        demanda_muestra = demanda_muestra.merge(factor_reponderacion, on="UBIGEO", how="left")
        demanda_muestra["POBLACION_CP"] = demanda_muestra["POBLACION_CP"] * demanda_muestra["FACTOR_REPONDERACION"]

    demanda_muestra["DEMAND_ID"] = range(len(demanda_muestra))
    demanda_muestra.to_parquet(cache_muestra)

R.log(f"Demanda tras muestreo estratificado por distrito (ponderado por poblacion): "
      f"{len(demanda_muestra)} / {len(demanda)} (fraccion={len(demanda_muestra)/len(demanda):.1%})")
print(demanda_muestra.groupby("DEP_NORM").size())


[17:27:42] Muestra de demanda ya cacheada -> se reutiliza


[17:27:42] Demanda tras muestreo estratificado por distrito (ponderado por poblacion): 5000 / 13358 (fraccion=37.4%)


DEP_NORM
JUNIN         2650
LAMBAYEQUE     858
LORETO        1492
dtype: int64


## 3. Extracción de la red vial (OSM) -- una sola pasada por el `.pbf`

Leer el `.pbf` completo (244 MB, todo el Perú) es lo caro (~2-3 min); no
depende de cuántos departamentos se pidan. Por eso se extraen los 3 a la
vez en una única pasada con `pyosmium`, cada uno filtrado por su bounding
box (+buffer, para no cortar rutas que cruzan el límite departamental).
Resultado cacheado por departamento en `data/raw/osm_extract/`.

In [4]:
boundaries = gpd.read_file(BOUNDARIES_PATH)
bboxes = {d: R.department_bbox(boundaries, d) for d in DEPARTAMENTOS}
raw_por_depto = R.extraer_redes_crudas(PBF_PATH, bboxes, OSM_EXTRACT_DIR)


[17:27:43] Extraccion cruda ya cacheada para todos los departamentos -- no se relee el .pbf


## 4. Loop principal por departamento

Para cada departamento y cada perfil (`car`, `foot`, `bike`):
1. Construir el grafo simplificado (intersecciones reales, no cada vértice
   de forma -- ver `src/routing._simplificar_a_intersecciones`).
2. Snapear demanda, resolutivos, candidatos I-3/I-4 y "cualquier categoría"
   a la red -- se reporta cuántos puntos no lograron engancharse y a qué
   distancia promedio los que sí.
3. Calcular las matrices/nearest requeridas, con Dijkstra sembrado desde el
   lado más chico (facilidades, no demanda) -- ver `nearest_optimo`.

Todo cacheado por departamento: si el parquet ya existe, no se recalcula.

In [5]:
reporte_snap = []
matriz_car_partes = []
matriz_car_candidatos_partes = []
nearest_foot_res_partes = []
nearest_bike_res_partes = []
nearest_foot_any_partes = []
detour_partes = []


def cache_o_calcula(path, fn):
    if os.path.exists(path):
        return pd.read_parquet(path)
    df = fn()
    df.to_parquet(path)
    return df


for dep in DEPARTAMENTOS:
    R.log(f"--- {dep} ---")
    raw = raw_por_depto[dep]

    dem_dep = demanda_muestra[demanda_muestra["DEP_NORM"] == dep]
    of_dep = oferta[oferta["DEPARTAMENTO_NORM"] == dep]
    res_dep = of_dep[of_dep["ES_RESOLUTIVO"]]
    urb_dep = dem_dep[dem_dep["ES_URBANO"]]

    R.log(f"  demanda muestreada={len(dem_dep)}  urbana={len(urb_dep)}  "
          f"oferta_total={len(of_dep)}  resolutiva={len(res_dep)}")

    def resumen_snap(nombre, perfil, snaps):
        oks = [s for s in snaps if s.ok]
        fallos = len(snaps) - len(oks)
        dist_media = float(np.mean([s.dist_m for s in oks])) if oks else float("nan")
        reporte_snap.append({
            "departamento": dep, "conjunto": nombre, "perfil": perfil,
            "n_puntos": len(snaps), "n_fallidos": fallos,
            "pct_fallidos": fallos / len(snaps) if snaps else np.nan,
            "dist_snap_media_m": dist_media,
        })
        R.log(f"  snap {nombre}/{perfil}: {fallos}/{len(snaps)} fallidos, dist media={dist_media:.0f}m")

    # ---------------- CAR: matriz completa demanda x resolutivo -------------
    t0 = time.time()
    g_car = R.construir_grafo(raw, "car")
    R.log(f"  grafo car: {g_car.number_of_nodes()} nodos, {g_car.number_of_edges()} edges", t0)

    dem_snap_car = R.snap_puntos(dem_dep.geometry.x.values, dem_dep.geometry.y.values, g_car)
    res_snap_car = R.snap_puntos(res_dep.geometry.x.values, res_dep.geometry.y.values, g_car)
    resumen_snap("demanda", "car", dem_snap_car)
    resumen_snap("resolutivo", "car", res_snap_car)

    matriz_dep = cache_o_calcula(
        os.path.join(PROCESSED_DIR, f"routing_matrix_car_{dep}.parquet"),
        lambda: R.construir_matriz_completa(
            dem_dep["DEMAND_ID"].values, dem_snap_car,
            res_dep["COD_IPRESS"].values, res_snap_car, g_car,
        ).assign(departamento=dep),
    )
    matriz_car_partes.append(matriz_dep)
    R.log(f"  matriz car {dep}: {len(matriz_dep)} pares")

    # --- Factor de detour: distancia ruteada vs. haversine (solo ROUTED) ---
    ruteados = matriz_dep[matriz_dep["estado"] == "ROUTED"].copy()
    if len(ruteados):
        dem_coords = pd.DataFrame({
            "lon_dem": dem_dep.geometry.x.values, "lat_dem": dem_dep.geometry.y.values,
        }, index=dem_dep["DEMAND_ID"].values)
        res_coords = pd.DataFrame({
            "lon_res": res_dep.geometry.x.values, "lat_res": res_dep.geometry.y.values,
        }, index=res_dep["COD_IPRESS"].values)
        ruteados = ruteados.merge(dem_coords, left_on="demand_id", right_index=True)
        ruteados = ruteados.merge(res_coords, left_on="facility_id", right_index=True)
        ruteados["dist_haversine_m"] = R.haversine_m(
            ruteados["lon_dem"], ruteados["lat_dem"], ruteados["lon_res"], ruteados["lat_res"],
        )
        ruteados = ruteados[ruteados["dist_haversine_m"] > 10]
        ruteados["factor_detour"] = ruteados["distance_m"] / ruteados["dist_haversine_m"]
        detour_partes.append(pd.DataFrame({
            "departamento": [dep], "n_pares": [len(ruteados)],
            "factor_detour_mediana": [ruteados["factor_detour"].median()],
            "factor_detour_media": [ruteados["factor_detour"].mean()],
            "factor_detour_p90": [ruteados["factor_detour"].quantile(0.9)],
        }))

    # --- Matriz demanda x candidatos I-3/I-4 (simulador de Fase 4) ---------
    # Sin esto solo se podria simular con establecimientos que YA son
    # resolutivos, lo cual no tiene sentido para un simulador de "ascenso".
    cand_dep = of_dep[of_dep["CATEGORIA_NORM"].isin(["I-3", "I-4"])]
    if len(cand_dep):
        cand_snap_car = R.snap_puntos(cand_dep.geometry.x.values, cand_dep.geometry.y.values, g_car)
        resumen_snap("candidatos_I3_I4", "car", cand_snap_car)
        matriz_cand_dep = cache_o_calcula(
            os.path.join(PROCESSED_DIR, f"routing_matrix_car_candidatos_{dep}.parquet"),
            lambda: R.construir_matriz_completa(
                dem_dep["DEMAND_ID"].values, dem_snap_car,
                cand_dep["COD_IPRESS"].values, cand_snap_car, g_car,
            ).assign(departamento=dep),
        )
        matriz_car_candidatos_partes.append(matriz_cand_dep)
        R.log(f"  matriz car candidatos I-3/I-4 {dep}: {len(matriz_cand_dep)} pares")

    # ---------------- FOOT: nearest resolutivo (comparacion vs. auto) -------
    t0 = time.time()
    g_foot = R.construir_grafo(raw, "foot")
    R.log(f"  grafo foot: {g_foot.number_of_nodes()} nodos, {g_foot.number_of_edges()} edges", t0)
    dem_snap_foot = R.snap_puntos(dem_dep.geometry.x.values, dem_dep.geometry.y.values, g_foot)
    res_snap_foot = R.snap_puntos(res_dep.geometry.x.values, res_dep.geometry.y.values, g_foot)
    resumen_snap("demanda", "foot", dem_snap_foot)
    resumen_snap("resolutivo", "foot", res_snap_foot)

    def _calc_nearest_foot_res():
        t0 = time.time()
        nearest_foot = R.nearest_optimo(g_foot, dem_dep["DEMAND_ID"].values, dem_snap_foot,
                                         res_dep["COD_IPRESS"].values, res_snap_foot, weight="travel_time_min")
        R.log(f"  nearest resolutivo a pie: {sum(1 for v in nearest_foot.values() if v)}/{len(nearest_foot)} alcanzables", t0)
        return pd.DataFrame([
            {"DEMAND_ID": k, "facility_id_foot": v[0] if v else None,
             "time_min_foot": v[1] if v else np.nan, "departamento": dep}
            for k, v in nearest_foot.items()
        ])

    nearest_foot_res_partes.append(cache_o_calcula(
        os.path.join(PROCESSED_DIR, f"nearest_foot_resolutivo_{dep}.parquet"), _calc_nearest_foot_res,
    ))

    todos_snap_foot = R.snap_puntos(of_dep.geometry.x.values, of_dep.geometry.y.values, g_foot)
    resumen_snap("cualquier_categoria", "foot", todos_snap_foot)

    if len(urb_dep):
        def _calc_nearest_foot_any():
            urb_snap_foot = R.snap_puntos(urb_dep.geometry.x.values, urb_dep.geometry.y.values, g_foot)
            t0 = time.time()
            nearest_any = R.nearest_optimo(g_foot, urb_dep["DEMAND_ID"].values, urb_snap_foot,
                                            of_dep["COD_IPRESS"].values, todos_snap_foot, weight="travel_time_min")
            R.log(f"  nearest cualquier-categoria a pie (urbano): "
                  f"{sum(1 for v in nearest_any.values() if v)}/{len(nearest_any)} alcanzables", t0)
            return pd.DataFrame([
                {"DEMAND_ID": k, "facility_id": v[0] if v else None,
                 "time_min_foot_cualquier_categoria": v[1] if v else np.nan, "departamento": dep}
                for k, v in nearest_any.items()
            ])

        nearest_foot_any_partes.append(cache_o_calcula(
            os.path.join(PROCESSED_DIR, f"nearest_foot_cualquier_categoria_{dep}.parquet"), _calc_nearest_foot_any,
        ))

    # ---------------- BIKE: nearest resolutivo (comparacion cruzada) -------
    t0 = time.time()
    g_bike = R.construir_grafo(raw, "bike")
    R.log(f"  grafo bike: {g_bike.number_of_nodes()} nodos, {g_bike.number_of_edges()} edges", t0)
    dem_snap_bike = R.snap_puntos(dem_dep.geometry.x.values, dem_dep.geometry.y.values, g_bike)
    res_snap_bike = R.snap_puntos(res_dep.geometry.x.values, res_dep.geometry.y.values, g_bike)
    resumen_snap("demanda", "bike", dem_snap_bike)
    resumen_snap("resolutivo", "bike", res_snap_bike)

    def _calc_nearest_bike():
        t0 = time.time()
        nearest_bike = R.nearest_optimo(g_bike, dem_dep["DEMAND_ID"].values, dem_snap_bike,
                                         res_dep["COD_IPRESS"].values, res_snap_bike, weight="travel_time_min")
        R.log(f"  nearest resolutivo en bici: {sum(1 for v in nearest_bike.values() if v)}/{len(nearest_bike)} alcanzables", t0)
        return pd.DataFrame([
            {"DEMAND_ID": k, "facility_id_bike": v[0] if v else None,
             "time_min_bike": v[1] if v else np.nan, "departamento": dep}
            for k, v in nearest_bike.items()
        ])

    nearest_bike_res_partes.append(cache_o_calcula(
        os.path.join(PROCESSED_DIR, f"nearest_bike_resolutivo_{dep}.parquet"), _calc_nearest_bike,
    ))


[17:27:47] --- LAMBAYEQUE ---


[17:27:47]   demanda muestreada=858  urbana=12  oferta_total=533  resolutiva=16


[17:27:50]   grafo car: 62063 nodos, 88426 edges (+3.5s)


[17:27:50]   snap demanda/car: 5/858 fallidos, dist media=506m


[17:27:50]   snap resolutivo/car: 0/16 fallidos, dist media=26m


...

[17:28:41]   nearest cualquier-categoria a pie (urbano): 16/19 alcanzables (+0.3s)


[17:28:48]   grafo bike: 87433 nodos, 116493 edges (+6.8s)


[17:28:48]   snap demanda/bike: 709/1492 fallidos, dist media=1373m


[17:28:48]   snap resolutivo/bike: 0/14 fallidos, dist media=29m


[17:28:49]   nearest resolutivo en bici: 142/1492 alcanzables (+0.5s)


## 5. Consolidación y export final

Se concatenan las piezas por departamento, se deriva "nearest" (mínimo por
punto de demanda) de la matriz completa, y se guarda todo lo que Fase 3 y
el dashboard (Fase 4) necesitan.

In [6]:
matriz_car = pd.concat(matriz_car_partes, ignore_index=True)
matriz_car.to_parquet(os.path.join(PROCESSED_DIR, "routing_matrix_car.parquet"))
nearest_car = R.nearest_from_matrix(matriz_car).rename(columns={"demand_id": "DEMAND_ID"})
nearest_car.to_parquet(os.path.join(PROCESSED_DIR, "nearest_car.parquet"))
R.log(f"Matriz car consolidada: {len(matriz_car)} pares | nearest_car: {len(nearest_car)} filas")

if matriz_car_candidatos_partes:
    matriz_car_candidatos = pd.concat(matriz_car_candidatos_partes, ignore_index=True)
    matriz_car_candidatos.to_parquet(os.path.join(PROCESSED_DIR, "routing_matrix_car_candidatos.parquet"))
    R.log(f"Matriz car candidatos I-3/I-4 consolidada: {len(matriz_car_candidatos)} pares")

nearest_foot_res = pd.concat(nearest_foot_res_partes, ignore_index=True)
nearest_foot_res.to_parquet(os.path.join(PROCESSED_DIR, "nearest_foot_resolutivo.parquet"))

nearest_bike_res = pd.concat(nearest_bike_res_partes, ignore_index=True)
nearest_bike_res.to_parquet(os.path.join(PROCESSED_DIR, "nearest_bike_resolutivo.parquet"))

if nearest_foot_any_partes:
    nearest_foot_any = pd.concat(nearest_foot_any_partes, ignore_index=True)
    nearest_foot_any.to_parquet(os.path.join(PROCESSED_DIR, "nearest_foot_cualquier_categoria.parquet"))

reporte_snap_df = pd.DataFrame(reporte_snap)
reporte_snap_df.to_csv(os.path.join(OUTPUTS_DIR, "reporte_snap.csv"), index=False)
print(reporte_snap_df.to_string(index=False))


[17:28:49] Matriz car consolidada: 103516 pares | nearest_car: 3562 filas


[17:28:49] Matriz car candidatos I-3/I-4 consolidada: 999848 pares


departamento            conjunto perfil  n_puntos  n_fallidos  pct_fallidos  dist_snap_media_m
  LAMBAYEQUE             demanda    car       858           5      0.005828         506.178178
  LAMBAYEQUE          resolutivo    car        16           0      0.000000          25.616800
  LAMBAYEQUE    candidatos_I3_I4    car       153           1      0.006536          37.140327
  LAMBAYEQUE             demanda   foot       858           6      0.006993         496.165397
  LAMBAYEQUE          resolutivo   foot        16           0      0.000000          18.381321
  LAMBAYEQUE cualquier_categoria   foot       533           1      0.001876          86.545812
  LAMBAYEQUE             demanda   bike       858           6      0.006993         496.902827
  LAMBAYEQUE          resolutivo   bike        16           0      0.000000          25.616800
       JUNIN             demanda    car      2650          68      0.025660         946.837088
       JUNIN          resolutivo    car        26 

## 6. Factor de detour (distancia en red vs. línea recta)

Este es el "documented fallback" que exige el enunciado: si algún punto
resultara no ruteable, cualquier estimación en línea recta debe justificarse
empíricamente contra datos ya ruteados -- no se inventa. Aquí se reporta el
factor real observado por departamento.

In [7]:
if detour_partes:
    detour_df = pd.concat(detour_partes, ignore_index=True)
    detour_df.to_csv(os.path.join(OUTPUTS_DIR, "reporte_detour_factor.csv"), index=False)
    print(detour_df.to_string(index=False))


departamento  n_pares  factor_detour_mediana  factor_detour_media  factor_detour_p90
  LAMBAYEQUE    13632               1.194502             1.234207           1.451586
       JUNIN    66404               1.463514             1.592690           2.133838
      LORETO      900               1.193955             1.231291           1.442269


## 7. Comparación entre modos (para Fase 3 / reporte)

Se compara, por punto de demanda, el establecimiento resolutivo más cercano
y su tiempo bajo auto/bici/a pie.

In [8]:
comparacion = nearest_car.merge(nearest_foot_res, on="DEMAND_ID", how="left") \
                          .merge(nearest_bike_res, on="DEMAND_ID", how="left")
comparacion.to_parquet(os.path.join(PROCESSED_DIR, "comparacion_modos.parquet"))

# Comparar como numeros: facility_id llega int64 (auto), facility_id_foot
# llega float64 (tiene NaN donde no hubo ruta a pie) -- comparar como texto
# los haria lucir "distintos" aunque sean el mismo establecimiento.
_id_car = pd.to_numeric(comparacion["facility_id"], errors="coerce")
_id_foot = pd.to_numeric(comparacion["facility_id_foot"], errors="coerce")
_ambos_ok = _id_car.notna() & _id_foot.notna()
n_cambia = (_id_car[_ambos_ok] != _id_foot[_ambos_ok]).sum()
R.log(f"Establecimiento mas cercano difiere entre auto y a pie en {n_cambia}/{_ambos_ok.sum()} puntos "
      f"con ruta valida en ambos modos ({n_cambia/_ambos_ok.sum():.1%})")


[17:28:49] Establecimiento mas cercano difiere entre auto y a pie en 414/3497 puntos con ruta valida en ambos modos (11.8%)


## 8. Log de ejecución

In [9]:
with open(os.path.join(LOGS_DIR, "fase2_routing.log"), "w", encoding="utf-8") as f:
    f.write(f"Ejecucion: {pd.Timestamp.now()}\n")
    f.write(f"Tiempo total: {time.time()-t_inicio:.1f}s\n")
    f.write(f"Demanda muestreada: {len(demanda_muestra)}/{len(demanda)}\n\n")
    f.write("=== Reporte de snap ===\n")
    f.write(reporte_snap_df.to_string(index=False))
    if detour_partes:
        f.write("\n\n=== Factor de detour ===\n")
        f.write(detour_df.to_string(index=False))

R.log(f"=== FASE 2 completa en {time.time()-t_inicio:.1f}s ===")


[17:28:50] === FASE 2 completa en 68.3s ===
